In [2]:
from pyspark.sql import functions as F

# 1. Lecture de la couche Bronze
df_bronze = spark.table("bronze_events")

# 2. Application des 5 règles d'or de Data Quality (Silver)
df_silver = (
    df_bronze
    # Règle 1 : Intégrité des sessions (suppression des sessions NULL)
    .filter(F.col("user_session").isNotNull())
    # Règle 2 : Élimination des achats fantômes (prix <= 0 pour un achat)
    .filter(~((F.col("event_type") == "purchase") & (F.col("price") <= 0)))
    # Règle 3 : Traitement des valeurs manquantes
    .fillna({"category_code": "Unknown", "brand": "Unknown"})
    # Règle 4 : Typage temporel strict en vrai Timestamp Spark
    .withColumn("event_time", F.to_timestamp(F.col("event_time")))
    # Règle 5 : Déduplication stricte
    .dropDuplicates()
)

# 3. Écriture Delta dans la couche Silver avec sécurité de schéma
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_events")
)

print(f"✅ Couche SILVER terminée : {spark.table('silver_events').count():,} lignes nettoyées.")

StatementMeta(, f81f9c97-0652-49ec-b6c9-478cef4f60be, 3, Finished, Available, Finished, False)

✅ Couche SILVER terminée : 15,664,083 lignes nettoyées.
